# Travel Reimbursement Approval Agent — Autonomous Compliance & Decision Engine

An enterprise-grade, agentic AI decision system designed to evaluate corporate travel reimbursement claims against company policy, receipt thresholds, and approval matrices. 

### Core Architecture Highlights
- **Agentic Workflow**: Powered by **LangGraph**, where an autonomous LLM Agent (**Google Gemini 3.5**) inspects unstructured claim context and dynamically selects compliance tools.
- **Deterministic Financial Precision**: Per-diem caps and deductions are computed using arbitrary-precision `Decimal` arithmetic (`ROUND_HALF_UP`) to guarantee zero financial hallucination.
- **Enterprise Guardrails & Fallback**: Hard policy rules and approval limits strictly enforce compliance. If API credentials are unavailable or network timeouts occur, the system automatically falls back to deterministic rule execution.
- **Strict Data Contracts**: Enforced via **Pydantic v2** (`strict=True`, `extra="forbid"`), producing an auditable 9-field output schema.

---

### Quick Start
1. **Prerequisites**: Python 3.10+
2. **Install Dependencies**:
   ```bash
   pip install "langgraph>=1.0" "pydantic>=2.0" "python-dotenv>=1.0" "google-genai>=1.0"
   ```
3. **Execution**: Run all cells in sequence. If `GOOGLE_API_KEY` is present in `.env`, the agent uses live LLM tool reasoning; otherwise, it seamlessly runs in deterministic fallback mode.


In [1]:
# One-time environment setup (run in a terminal, not on every notebook execution):
# python -m pip install "langgraph>=1.0" "pydantic>=2.0" "python-dotenv>=1.0"
# Optional, only for RUN_LLM_REVIEW=true:
# python -m pip install "google-genai>=1.0"

import importlib.util

required_packages = {"pydantic": "pydantic", "langgraph": "langgraph", "dotenv": "python-dotenv"}
missing_packages = [distribution for module, distribution in required_packages.items() if importlib.util.find_spec(module) is None]
if missing_packages:
    raise RuntimeError("Install once before opening this notebook: python -m pip install " + " ".join(missing_packages))
print("Dependency preflight passed; no packages were installed by this notebook.")


Dependency preflight passed; no packages were installed by this notebook.


## 1. System Configuration & Runtime Initialization

Loads environment variables, configures LLM parameters (model, timeout, concurrency), and initializes system logging. The agent defaults to `gemini-3.5-flash-lite` for ultra-low latency tool selection.


In [2]:
import asyncio
import importlib.util
import json
import os
import re
import time
from collections import Counter
from datetime import date
from decimal import Decimal, ROUND_HALF_UP
from typing import Any, Literal, TypedDict

from dotenv import load_dotenv
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, ConfigDict, Field, ValidationError, field_validator, model_validator

load_dotenv(override=False)
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY") or os.getenv("Travel_Agent_Gemini_API_Key")
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.5-flash-lite")
LLM_TIMEOUT_SECONDS = float(os.getenv("LLM_TIMEOUT_SECONDS", "10"))
RUN_LLM_AGENT = os.getenv("RUN_LLM_AGENT", "true").strip().lower() == "true"
MAX_CONCURRENT_CLAIMS = int(os.getenv("MAX_CONCURRENT_CLAIMS", "2"))

if LLM_TIMEOUT_SECONDS <= 0 or LLM_TIMEOUT_SECONDS > 30:
    raise ValueError("LLM_TIMEOUT_SECONDS must be between 0 and 30")
if not 1 <= MAX_CONCURRENT_CLAIMS <= 5:
    raise ValueError("MAX_CONCURRENT_CLAIMS must be between 1 and 5")

print(f"Agent mode: {'LLM-assisted agent (Gemini ' + GEMINI_MODEL + ') with deterministic fallback' if RUN_LLM_AGENT and GEMINI_API_KEY else 'deterministic_policy fallback'}")
print(f"Configured concurrency: {MAX_CONCURRENT_CLAIMS}; LLM Agent active: {bool(RUN_LLM_AGENT and GEMINI_API_KEY)}")


Agent mode: LLM-assisted agent (Gemini gemini-3.5-flash-lite) with deterministic fallback
Configured concurrency: 2; LLM Agent active: True


## 2. Domain Data Contracts & Test Dataset

Defines strict Pydantic v2 models for ingestion (`ClaimInput`, `LineItem`), intermediate agent assessment (`AgentAssessment`), and the final public output (`ClaimResult`). 
Monetary values are constrained to non-negative floats, dates must be valid ISO 8601 strings, and trip date consistency is validated at ingestion.


In [3]:
class LineItem(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)
    category: str
    description: str
    amount: float = Field(ge=0)
    receipt_attached: bool
    itemized_receipt: bool | None = None
    quantity: int | None = Field(default=None, gt=0)
    unit: Literal["day", "night"] | None = None
    airfare_class: Literal["economy", "business", "first"] | None = None

    @model_validator(mode="after")
    def validate_quantity(self):
        if self.unit is not None and self.quantity is None:
            raise ValueError("A capped line with a unit must include its quantity")
        return self


class ClaimInput(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)
    claim_id: str
    scenario: str
    employee: str
    trip_start: str
    trip_end: str
    submitted_date: str
    business_purpose: str
    business_purpose_documented: bool
    line_items: list[LineItem]

    @field_validator("trip_start", "trip_end", "submitted_date")
    @classmethod
    def validate_iso_date(cls, value: str) -> str:
        date.fromisoformat(value)
        return value

    @model_validator(mode="after")
    def validate_trip_range(self):
        if date.fromisoformat(self.trip_end) < date.fromisoformat(self.trip_start):
            raise ValueError("trip_end cannot precede trip_start")
        if not self.line_items:
            raise ValueError("A claim must include at least one line item")
        return self


class AgentAssessment(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)
    decision: Literal["APPROVE", "PARTIAL_APPROVE", "REJECT", "MANUAL_REVIEW"]
    policy_refs: list[str]
    explanation: str
    ambiguities: list[str] = Field(default_factory=list)


class ClaimResult(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)
    claim_id: str
    decision: Literal["APPROVE", "PARTIAL_APPROVE", "REJECT", "MANUAL_REVIEW"]
    approved_amount: float | None
    deducted_amount: float | None
    missing_docs: list[str]
    policy_refs: list[str]
    confidence: float = Field(ge=0.0, le=1.0)
    explanation: str
    tools_used: list[str]

    @model_validator(mode="after")
    def validate_amount_semantics(self):
        if self.decision == "MANUAL_REVIEW":
            if self.approved_amount is not None or self.deducted_amount is not None:
                raise ValueError("MANUAL_REVIEW requires null for both amount fields")
        elif self.approved_amount is None or self.deducted_amount is None:
            raise ValueError("Only MANUAL_REVIEW may use null financial amounts")
        if self.decision == "APPROVE" and self.deducted_amount != 0:
            raise ValueError("APPROVE must not include a deduction")
        if self.decision == "REJECT" and self.approved_amount != 0:
            raise ValueError("REJECT must have approved_amount 0")
        return self


RAW_CLAIMS = [
    {
        "claim_id": "CLM-001", "scenario": "Attend 2-day industry conference (business)", "employee": "A. Rivera",
        "trip_start": "2026-06-10", "trip_end": "2026-06-12", "submitted_date": "2026-06-20",
        "business_purpose": "Attend a 2-day industry conference (business)", "business_purpose_documented": True,
        "line_items": [
            {"category": "airfare", "description": "Round-trip economy airfare", "amount": 420.00, "receipt_attached": True, "itemized_receipt": True, "airfare_class": "economy"},
            {"category": "lodging", "description": "Hotel, 2 nights @ $180", "amount": 360.00, "receipt_attached": True, "itemized_receipt": True, "quantity": 2, "unit": "night"},
            {"category": "meals", "description": "Meals, 3 days @ ~$60/day", "amount": 180.00, "receipt_attached": True, "itemized_receipt": True, "quantity": 3, "unit": "day"},
            {"category": "conference_fees", "description": "Conference registration", "amount": 150.00, "receipt_attached": True, "itemized_receipt": True},
        ],
    },
    {
        "claim_id": "CLM-002", "scenario": "Weekend hotel stay", "employee": "B. Osei",
        "trip_start": "2026-06-14", "trip_end": "2026-06-15", "submitted_date": "2026-06-25",
        "business_purpose": "Weekend hotel stay; no documented business purpose", "business_purpose_documented": False,
        "line_items": [
            {"category": "spa", "description": "Hotel spa package", "amount": 300.00, "receipt_attached": True, "itemized_receipt": True},
            {"category": "minibar", "description": "In-room minibar", "amount": 80.00, "receipt_attached": True, "itemized_receipt": True},
        ],
    },
    {
        "claim_id": "CLM-003", "scenario": "Client site visit (business)", "employee": "C. Nakamura",
        "trip_start": "2026-06-08", "trip_end": "2026-06-10", "submitted_date": "2026-06-22",
        "business_purpose": "Client site visit (business)", "business_purpose_documented": True,
        "line_items": [
            {"category": "airfare", "description": "Round-trip economy airfare", "amount": 300.00, "receipt_attached": True, "itemized_receipt": True, "airfare_class": "economy"},
            {"category": "lodging", "description": "Hotel, 2 nights @ $250", "amount": 500.00, "receipt_attached": True, "itemized_receipt": True, "quantity": 2, "unit": "night"},
            {"category": "meals", "description": "Meals, 2 days @ $70/day", "amount": 140.00, "receipt_attached": True, "itemized_receipt": True, "quantity": 2, "unit": "day"},
        ],
    },
    {
        "claim_id": "CLM-004", "scenario": "International vendor negotiation (business)", "employee": "D. Fischer",
        "trip_start": "2026-06-16", "trip_end": "2026-06-18", "submitted_date": "2026-06-28",
        "business_purpose": "International vendor negotiation (business)", "business_purpose_documented": True,
        "line_items": [
            {"category": "airfare", "description": "Business-class international airfare", "amount": 2400.00, "receipt_attached": True, "itemized_receipt": True, "airfare_class": "business"},
            {"category": "lodging", "description": "Hotel, 3 nights", "amount": 600.00, "receipt_attached": False, "itemized_receipt": None, "quantity": 3, "unit": "night"},
        ],
    },
    {
        "claim_id": "CLM-005", "scenario": "Client dinner / business development", "employee": "E. Haddad",
        "trip_start": "2026-06-11", "trip_end": "2026-06-11", "submitted_date": "2026-06-24",
        "business_purpose": "Client dinner for 4 (business development)", "business_purpose_documented": True,
        "line_items": [
            {"category": "meals", "description": "Client dinner for 4 (business development)", "amount": 220.00, "receipt_attached": False, "itemized_receipt": None, "quantity": 1, "unit": "day"},
        ],
    },
]

CLAIMS = {item["claim_id"]: ClaimInput.model_validate(item).model_dump(mode="json") for item in RAW_CLAIMS}
print(f"Validated {len(CLAIMS)} claims: {', '.join(CLAIMS)}")






Validated 5 claims: CLM-001, CLM-002, CLM-003, CLM-004, CLM-005


## 3. Deterministic Policy Tools & Arithmetic Engine

Implements four modular, auditable domain functions:
1. `lookup_policy`: Resolves active `POL-*` policy clauses and requirements.
2. `check_receipts`: Verifies itemized receipt attachments for line items >$25 and always-required categories (airfare, lodging).
3. `calculate_reimbursement`: Computes line-item eligibility and per-diem caps (Meals: $75/day, Lodging: $200/night, Transport: $50/day) using `Decimal` math.
4. `check_claim_timing_and_thresholds`: Verifies the 30-day submission window and establishes the authorization tier ($500 auto-approve / $2000 manager / >$2000 director review).


In [4]:
POLICY = {
    "POL-CAT-01": "Eligible business expenses: economy airfare, lodging, meals, ground transport, and conference/registration fees; documented business purpose is required.",
    "POL-CAT-02": "Alcohol/minibar, spa/gym/personal entertainment, personal shopping/gifts, penalties, and any personal non-business expense are not reimbursable; deduct in full.",
    "POL-PD-01": "Meals: maximum $75 per day; reimburse up to the cap and deduct excess.",
    "POL-PD-02": "Lodging: maximum $200 per night; reimburse up to the cap and deduct excess.",
    "POL-PD-03": "Ground transport: maximum $50 per day; reimburse up to the cap and deduct excess.",
    "POL-AIR-01": "Only economy airfare is reimbursable. Business/first class is an exception requiring Manual Review.",
    "POL-RCT-01": "Every line item greater than $25 requires an attached itemized receipt; airfare and lodging always require a receipt.",
    "POL-RCT-02": "A missing required receipt routes the claim to Manual Review; do not silently reject the item.",
    "POL-APR-01": "After per-diem deductions, total <= $500 is the auto-approval tier when compliant.",
    "POL-APR-02": "After per-diem deductions, $500 < total <= $2,000 is the manager tier; a compliant claim may receive an APPROVE recommendation.",
    "POL-APR-03": "After per-diem deductions, total > $2,000 requires Manual Review / director approval.",
    "POL-TIME-01": "Submission more than 30 days after an expense date routes to Manual Review.",
}
INELIGIBLE_CATEGORIES = {"alcohol", "minibar", "spa", "gym", "personal_entertainment", "personal_shopping", "gifts", "traffic_fine", "penalty", "late_fee", "personal"}
ELIGIBLE_CATEGORIES = {"airfare", "lodging", "meals", "ground_transport", "conference_fees"}
CAPS = {"meals": Decimal("75"), "lodging": Decimal("200"), "ground_transport": Decimal("50")}
TOOL_NAMES = ["lookup_policy", "check_receipts", "calculate_reimbursement", "check_claim_timing_and_thresholds"]


def money(value: Any) -> Decimal:
    return Decimal(str(value)).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)


def as_amount(value: Decimal) -> float:
    return float(value.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP))


def claim_total(claim: dict[str, Any]) -> Decimal:
    return sum((money(line["amount"]) for line in claim["line_items"]), Decimal("0.00"))


def calculate_reimbursement(claim: dict[str, Any]) -> dict[str, Any]:
    line_results, review_reasons, policy_refs = [], [], set()
    candidate_total, deducted_total = Decimal("0.00"), Decimal("0.00")
    for line in claim["line_items"]:
        category = line["category"].strip().lower()
        amount = money(line["amount"])
        candidate, deduction = amount, Decimal("0.00")
        line_refs = set()
        if category in INELIGIBLE_CATEGORIES:
            candidate, deduction = Decimal("0.00"), amount
            line_refs.add("POL-CAT-02")
        elif category not in ELIGIBLE_CATEGORIES:
            policy_refs.update({"POL-CAT-01", "POL-CAT-02"})
            review_reasons.append(f"Unknown expense category: {line['category']}")
        else:
            line_refs.add("POL-CAT-01")
            if category == "airfare":
                line_refs.add("POL-AIR-01")
                if line.get("airfare_class") in {"business", "first"}:
                    review_reasons.append(f"{line.get('airfare_class').title()}-class airfare exception")
            if category in CAPS:
                unit = "night" if category == "lodging" else "day"
                line_refs.add({"meals": "POL-PD-01", "lodging": "POL-PD-02", "ground_transport": "POL-PD-03"}[category])
                quantity = line.get("quantity")
                if quantity is None:
                    review_reasons.append(f"Missing {unit} count for {category} cap")
                else:
                    candidate = min(amount, CAPS[category] * Decimal(quantity))
                    deduction = amount - candidate
        if line["category"] in {"airfare", "lodging"} or amount > Decimal("25"):
            line_refs.add("POL-RCT-01")
        candidate_total += candidate
        deducted_total += deduction
        policy_refs.update(line_refs)
        line_results.append({
            "category": line["category"], "description": line["description"],
            "claimed": as_amount(amount), "candidate_reimbursement": as_amount(candidate),
            "candidate_deduction": as_amount(deduction), "policy_refs": sorted(line_refs),
        })
    eligible_lines = [line for line in claim["line_items"] if line["category"].strip().lower() in ELIGIBLE_CATEGORIES]
    if eligible_lines and (not claim["business_purpose_documented"] or not claim["business_purpose"].strip()):
        review_reasons.append("Documented business purpose is missing for eligible expenses")
    return {
        "line_items": line_results, "candidate_reimbursement": as_amount(candidate_total),
        "candidate_deduction": as_amount(deducted_total), "claimed_total": as_amount(claim_total(claim)),
        "review_reasons": list(dict.fromkeys(review_reasons)), "policy_refs": sorted(policy_refs),
    }


def lookup_policy(claim_id: str) -> dict[str, Any]:
    calc = calculate_reimbursement(CLAIMS[claim_id])
    refs = set(calc["policy_refs"]) | {"POL-TIME-01"}
    total = money(calc["candidate_reimbursement"])
    refs.add("POL-APR-01" if total <= Decimal("500") else "POL-APR-02" if total <= Decimal("2000") else "POL-APR-03")
    return {"claim_id": claim_id, "policy_refs": sorted(refs), "rules": {ref: POLICY[ref] for ref in sorted(refs)}}


def check_receipts(claim_id: str) -> dict[str, Any]:
    missing, refs = [], {"POL-RCT-01"}
    for line in CLAIMS[claim_id]["line_items"]:
        amount = money(line["amount"])
        required = line["category"] in {"airfare", "lodging"} or amount > Decimal("25")
        if not required:
            continue
        if not line["receipt_attached"]:
            label = "Airfare receipt" if line["category"] == "airfare" else "Lodging receipt" if line["category"] == "lodging" else f"Itemized {line['category']} receipt"
            missing.append(label)
        elif amount > Decimal("25") and line.get("itemized_receipt") is not True:
            missing.append(f"Itemized {line['category']} receipt")
    if missing:
        refs.add("POL-RCT-02")
    return {"claim_id": claim_id, "missing_docs": list(dict.fromkeys(missing)), "policy_refs": sorted(refs)}


def check_claim_timing_and_thresholds(claim_id: str) -> dict[str, Any]:
    claim, calc = CLAIMS[claim_id], calculate_reimbursement(CLAIMS[claim_id])
    submitted = date.fromisoformat(claim["submitted_date"])
    start, end = date.fromisoformat(claim["trip_start"]), date.fromisoformat(claim["trip_end"])
    age_from_start, age_from_end = (submitted - start).days, (submitted - end).days
    if submitted < start:
        timing = "conflicting_dates"
    elif age_from_start <= 30:
        timing = "within_window"
    elif age_from_end > 30:
        timing = "late"
    else:
        timing = "uncertain_trip_range"
    total = money(calc["candidate_reimbursement"])
    if total <= Decimal("500"):
        tier, tier_ref = "auto_approval", "POL-APR-01"
    elif total <= Decimal("2000"):
        tier, tier_ref = "manager_approval", "POL-APR-02"
    else:
        tier, tier_ref = "director_manual_review", "POL-APR-03"
    reasons = []
    if timing == "late":
        reasons.append("Submission is more than 30 days after the expense period")
    elif timing != "within_window":
        reasons.append("Expense date cannot be safely established from the supplied trip dates")
    if tier == "director_manual_review":
        reasons.append("Candidate reimbursable total exceeds $2,000")
    return {
        "claim_id": claim_id, "timeliness": timing, "days_after_trip_start": age_from_start,
        "days_after_trip_end": age_from_end, "approval_tier": tier,
        "candidate_reimbursement": as_amount(total), "review_reasons": reasons,
        "policy_refs": ["POL-TIME-01", tier_ref],
    }


def dispatch_tool(name: str, args: dict[str, Any]) -> dict[str, Any]:
    claim_id = args.get("claim_id")
    if claim_id not in CLAIMS:
        return {"error": "Unknown claim_id; no claim data was accessed."}
    functions = {
        "lookup_policy": lookup_policy,
        "check_receipts": check_receipts,
        "calculate_reimbursement": lambda value: calculate_reimbursement(CLAIMS[value]),
        "check_claim_timing_and_thresholds": check_claim_timing_and_thresholds,
    }
    if name not in functions:
        return {"error": "Unsupported tool name; no tool was executed."}
    return functions[name](claim_id)


TOOL_DECLARATIONS = [
    {"type": "function", "name": "lookup_policy", "description": "Retrieve relevant policy rules and stable IDs for this claim.", "parameters": {"type": "object", "properties": {"claim_id": {"type": "string"}}, "required": ["claim_id"]}},
    {"type": "function", "name": "check_receipts", "description": "Check receipt requirements and missing documents for this claim.", "parameters": {"type": "object", "properties": {"claim_id": {"type": "string"}}, "required": ["claim_id"]}},
    {"type": "function", "name": "calculate_reimbursement", "description": "Calculate candidate reimbursement and policy deductions per line. Results remain provisional until decision guardrails run.", "parameters": {"type": "object", "properties": {"claim_id": {"type": "string"}}, "required": ["claim_id"]}},
    {"type": "function", "name": "check_claim_timing_and_thresholds", "description": "Check the 30-day submission window and approval tier.", "parameters": {"type": "object", "properties": {"claim_id": {"type": "string"}}, "required": ["claim_id"]}},
]






## 4. Agentic Workflow Architecture (LangGraph + Gemini Tool Calling)

The core decision graph is orchestrated using **LangGraph**:
```
START ──► prepare ──► agent_reasoning ──► execute_tools ──► finalize ──► END
```
- **`prepare_node`**: Loads and validates claim state.
- **`agent_reasoning_node`**: Autonomous Gemini LLM agent reviews the claim context, formulates an audit hypothesis, and outputs structured tool selections.
- **`execute_tools_node`**: Invokes the selected tools deterministically, ensuring mandatory compliance tools are always executed.
- **`finalize_node`**: Enforces strict financial guardrails. Claims with policy exceptions, missing receipts, or amounts >$2,000 route to `MANUAL_REVIEW` (with `approved_amount = null`).


In [5]:
class ClaimState(TypedDict, total=False):
    claim_id: str
    claim: dict[str, Any]
    agent_reasoning: str
    tools_selected_by_agent: list[str]
    tool_outputs: dict[str, Any]
    tools_used: list[str]
    execution_mode: str
    result: dict[str, Any]
    audit: dict[str, Any]


def prepare_node(state: ClaimState) -> dict[str, Any]:
    claim_id = state["claim_id"]
    if claim_id not in CLAIMS:
        raise KeyError(f"Unknown claim ID: {claim_id}")
    return {"claim": CLAIMS[claim_id], "execution_mode": "deterministic_policy"}


def is_transient_error(exc: Exception) -> bool:
    status = getattr(exc, "status_code", None) or getattr(exc, "code", None)
    try:
        status = int(getattr(status, "value", status))
    except (TypeError, ValueError):
        status = None
    return status in {408, 429, 500, 502, 503, 504} or isinstance(exc, (TimeoutError, ConnectionError, asyncio.TimeoutError))


def llm_select_tools(claim: dict[str, Any]) -> tuple[list[str], str, str]:
    """Ask Gemini to inspect the claim and decide which tools to call."""
    if not (RUN_LLM_AGENT and GEMINI_API_KEY):
        return list(TOOL_NAMES), "LLM disabled or API key missing; default compliance toolset selected.", "deterministic_policy"
    if importlib.util.find_spec("google.genai") is None:
        return list(TOOL_NAMES), "google-genai library not installed; fallback toolset selected.", "deterministic_policy"

    from google import genai
    from google.genai import types

    tools_doc = [
        {"name": "lookup_policy", "description": "Lookup reimbursement policy clauses (POL-*) and category guidelines."},
        {"name": "check_receipts", "description": "Verify receipt attachments for expenses >$25, airfare, and lodging."},
        {"name": "calculate_reimbursement", "description": "Calculate itemized eligible amounts, deductions, and per-diem caps."},
        {"name": "check_claim_timing_and_thresholds", "description": "Check 30-day submission timeliness and approval tiers ($500/$2000)."}
    ]
    prompt = f"""You are an autonomous AI Travel Reimbursement Agent.
Evaluate the claim below and decide which tools are required to audit it thoroughly.

Claim to Evaluate:
- ID: {claim['claim_id']}
- Purpose: {claim['business_purpose']}
- Trip: {claim['trip_start']} to {claim['trip_end']} (Submitted: {claim['submitted_date']})
- Line Items: {json.dumps(claim['line_items'])}

Available Tools:
{json.dumps(tools_doc, indent=2)}

Respond with a JSON object containing:
- reasoning: Short explanation of what needs to be verified.
- tools_to_call: Array of tool names from the available list that should be invoked.
"""
    client = genai.Client(api_key=GEMINI_API_KEY)
    for attempt in range(2):
        try:
            response = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    temperature=0.0
                )
            )
            parsed = json.loads(response.text.strip())
            tools = [t for t in parsed.get("tools_to_call", []) if t in TOOL_NAMES]
            if len(tools) >= 2:
                return tools, parsed.get("reasoning", "LLM tool selection completed."), "llm_agentic"
            break
        except Exception as exc:
            if attempt == 0 and is_transient_error(exc):
                time.sleep(0.5)
                continue
            break
    return list(TOOL_NAMES), "LLM tool call error or incomplete selection; safety fallback applied.", "deterministic_policy"


def agent_reasoning_node(state: ClaimState) -> dict[str, Any]:
    claim = state["claim"]
    tools, reasoning, mode = llm_select_tools(claim)
    return {
        "tools_selected_by_agent": tools,
        "agent_reasoning": reasoning,
        "execution_mode": mode
    }


def execute_tools_node(state: ClaimState) -> dict[str, Any]:
    claim_id = state["claim_id"]
    tools_to_run = set(state.get("tools_selected_by_agent") or TOOL_NAMES)
    tools_to_run.update(TOOL_NAMES)
    ordered_tools = [t for t in TOOL_NAMES if t in tools_to_run]
    outputs = {name: dispatch_tool(name, {"claim_id": claim_id}) for name in ordered_tools}
    return {"tool_outputs": outputs, "tools_used": ordered_tools}


def build_public_result(state: ClaimState) -> tuple[dict[str, Any], dict[str, Any]]:
    claim, outputs = state["claim"], state["tool_outputs"]
    calculation = outputs["calculate_reimbursement"]
    receipt_check = outputs["check_receipts"]
    tier_check = outputs["check_claim_timing_and_thresholds"]
    policy_data = outputs["lookup_policy"]

    review_reasons = list(dict.fromkeys(
        list(calculation["review_reasons"])
        + list(tier_check["review_reasons"])
        + (["A required receipt is missing"] if receipt_check["missing_docs"] else [])
    ))
    candidate, deductions = money(calculation["candidate_reimbursement"]), money(calculation["candidate_deduction"])
    if review_reasons:
        decision, approved_amount, deducted_amount = "MANUAL_REVIEW", None, None
    elif candidate <= 0:
        decision, approved_amount, deducted_amount = "REJECT", 0.0, as_amount(deductions)
    elif deductions > 0:
        decision, approved_amount, deducted_amount = "PARTIAL_APPROVE", as_amount(candidate), as_amount(deductions)
    else:
        decision, approved_amount, deducted_amount = "APPROVE", as_amount(candidate), 0.0

    refs = set(policy_data["policy_refs"]) | set(receipt_check["policy_refs"]) | set(calculation["policy_refs"]) | set(tier_check["policy_refs"])
    if receipt_check["missing_docs"]:
        refs.add("POL-RCT-02")
    if any("airfare exception" in reason.lower() for reason in review_reasons):
        refs.add("POL-AIR-01")
    if any("business purpose" in reason.lower() for reason in review_reasons):
        refs.add("POL-CAT-01")

    if decision == "APPROVE":
        explanation = f"All claimed items meet the supplied policy checks. Candidate reimbursement is ${as_amount(candidate):.2f}."
        if tier_check["approval_tier"] == "manager_approval":
            explanation += " This is the manager approval tier (POL-APR-02); the recommendation is APPROVE."
    elif decision == "PARTIAL_APPROVE":
        lines = [item for item in calculation["line_items"] if item["candidate_deduction"] > 0]
        details = "; ".join(f"{item['category']} deduction ${item['candidate_deduction']:.2f}" for item in lines)
        explanation = f"Reimburse up to applicable limits: ${as_amount(candidate):.2f} approved and ${as_amount(deductions):.2f} deducted. {details}."
    elif decision == "REJECT":
        explanation = f"No claimed amount is reimbursable. The ineligible items are deducted in full (${as_amount(deductions):.2f}) under POL-CAT-02."
    else:
        explanation = "Manual Review is required: " + "; ".join(review_reasons) + f". The original ${calculation['claimed_total']:.2f} claim remains pending; no final approval or deduction has been determined."

    confidence = 0.55 if any("unknown expense category" in reason.lower() or "cannot be safely" in reason.lower() for reason in review_reasons) else (0.96 if decision == "PARTIAL_APPROVE" else 0.98)
    result = ClaimResult(
        claim_id=claim["claim_id"], decision=decision, approved_amount=approved_amount,
        deducted_amount=deducted_amount, missing_docs=receipt_check["missing_docs"],
        policy_refs=sorted(refs), confidence=confidence, explanation=explanation,
        tools_used=list(state["tools_used"]),
    ).model_dump(mode="json")
    audit = {
        "claim_id": claim["claim_id"], "execution_mode": state["execution_mode"],
        "agent_reasoning": state.get("agent_reasoning", ""),
        "tools_selected_by_agent": state.get("tools_selected_by_agent", []),
        "review_reasons": review_reasons,
        "provisional_candidate_reimbursement": as_amount(candidate),
        "provisional_candidate_deduction": as_amount(deductions),
        "original_claim_total_pending": calculation["claimed_total"] if decision == "MANUAL_REVIEW" else 0.0,
        "tool_outputs": outputs,
    }
    return result, audit


def finalize_node(state: ClaimState) -> dict[str, Any]:
    result, audit = build_public_result(state)
    return {"result": result, "audit": audit}


def build_graph():
    graph = StateGraph(ClaimState)
    graph.add_node("prepare", prepare_node)
    graph.add_node("agent_reasoning", agent_reasoning_node)
    graph.add_node("execute_tools", execute_tools_node)
    graph.add_node("finalize", finalize_node)
    graph.add_edge(START, "prepare")
    graph.add_edge("prepare", "agent_reasoning")
    graph.add_edge("agent_reasoning", "execute_tools")
    graph.add_edge("execute_tools", "finalize")
    graph.add_edge("finalize", END)
    return graph.compile()


agent_graph = build_graph()
print("Agentic LangGraph workflow (LLM Tool Calling + Guardrails) compiled.")


Agentic LangGraph workflow (LLM Tool Calling + Guardrails) compiled.


## 5. Offline Self-Checks & Guardrail Verification

Automated unit tests verifying:
- Schema rejection of malformed outputs (`ValidationError`).
- Safe escalation to `MANUAL_REVIEW` when encountering unknown categories.
- Accuracy of decisions, per-diem cap deductions, and exact schema contracts across all claims.


In [6]:
try:
    ClaimResult.model_validate({"claim_id": "bad", "decision": "APPROVE"})
except ValidationError:
    invalid_result_rejected = True
else:
    invalid_result_rejected = False
assert invalid_result_rejected

ambiguous_fixture = dict(CLAIMS["CLM-001"])
ambiguous_fixture["line_items"] = [dict(CLAIMS["CLM-001"]["line_items"][0], category="unknown_expense")]
original_claim = CLAIMS["CLM-001"]
CLAIMS["CLM-001"] = ambiguous_fixture
try:
    ambiguous_run = agent_graph.invoke({"claim_id": "CLM-001"})
finally:
    CLAIMS["CLM-001"] = original_claim
assert ambiguous_run["result"]["decision"] == "MANUAL_REVIEW"
assert ambiguous_run["result"]["approved_amount"] is None and ambiguous_run["result"]["deducted_amount"] is None

expected = {
    "CLM-001": ("APPROVE", 1110.0, 0.0),
    "CLM-002": ("REJECT", 0.0, 380.0),
    "CLM-003": ("PARTIAL_APPROVE", 840.0, 100.0),
    "CLM-004": ("MANUAL_REVIEW", None, None),
    "CLM-005": ("MANUAL_REVIEW", None, None),
}
started = time.perf_counter()
smoke_runs = [agent_graph.invoke({"claim_id": claim_id}) for claim_id in CLAIMS]
elapsed = time.perf_counter() - started
for run in smoke_runs:
    result = run["result"]
    assert (result["decision"], result["approved_amount"], result["deducted_amount"]) == expected[result["claim_id"]]
    assert set(result) == {"claim_id", "decision", "approved_amount", "deducted_amount", "missing_docs", "policy_refs", "confidence", "explanation", "tools_used"}
    assert len(result["tools_used"]) >= 2
    assert run["audit"]["execution_mode"] in {"llm_agentic", "deterministic_policy"}
print(f"Self-checks passed in {elapsed:.3f}s: validation, safe escalation, expected outcomes, and schema.")


Self-checks passed in 16.538s: validation, safe escalation, expected outcomes, and schema.


## 6. End-to-End Evaluation of Candidate Claims

Processes all 5 assignment claims through the live LangGraph agentic workflow. Each claim dynamically records the LLM agent's rationale, tools selected, and final disposition.


In [7]:
started = time.perf_counter()
claim_runs = []
for index, claim_id in enumerate(CLAIMS, start=1):
    print(f"[{index}/{len(CLAIMS)}] Evaluating {claim_id} with Agentic Workflow...", flush=True)
    run = agent_graph.invoke({"claim_id": claim_id})
    claim_runs.append(run)
    audit = run["audit"]
    print(f"[{index}/{len(CLAIMS)}] Finished {claim_id}: {run['result']['decision']} (mode={audit['execution_mode']}, tools={audit.get('tools_selected_by_agent')})", flush=True)

evaluation_seconds = time.perf_counter() - started
results = [run["result"] for run in claim_runs]
audits = [run["audit"] for run in claim_runs]

expected = {
    "CLM-001": ("APPROVE", 1110.0, 0.0),
    "CLM-002": ("REJECT", 0.0, 380.0),
    "CLM-003": ("PARTIAL_APPROVE", 840.0, 100.0),
    "CLM-004": ("MANUAL_REVIEW", None, None),
    "CLM-005": ("MANUAL_REVIEW", None, None),
}
for result in results:
    assert (result["decision"], result["approved_amount"], result["deducted_amount"]) == expected[result["claim_id"]]
    ClaimResult.model_validate(result)
    assert len(result) == 9 and len(result["tools_used"]) >= 2

print(f"Validated five claims in {evaluation_seconds:.3f}s; Agentic evaluation complete.")


[1/5] Evaluating CLM-001 with Agentic Workflow...
[1/5] Finished CLM-001: APPROVE (mode=llm_agentic, tools=['lookup_policy', 'check_receipts', 'check_claim_timing_and_thresholds', 'calculate_reimbursement'])
[2/5] Evaluating CLM-002 with Agentic Workflow...
[2/5] Finished CLM-002: REJECT (mode=llm_agentic, tools=['lookup_policy', 'check_receipts', 'calculate_reimbursement', 'check_claim_timing_and_thresholds'])
[3/5] Evaluating CLM-003 with Agentic Workflow...
[3/5] Finished CLM-003: PARTIAL_APPROVE (mode=llm_agentic, tools=['lookup_policy', 'check_receipts', 'calculate_reimbursement', 'check_claim_timing_and_thresholds'])
[4/5] Evaluating CLM-004 with Agentic Workflow...
[4/5] Finished CLM-004: MANUAL_REVIEW (mode=llm_agentic, tools=['lookup_policy', 'check_receipts', 'calculate_reimbursement', 'check_claim_timing_and_thresholds'])
[5/5] Evaluating CLM-005 with Agentic Workflow...
[5/5] Finished CLM-005: MANUAL_REVIEW (mode=llm_agentic, tools=['lookup_policy', 'check_receipts', 'check

## 7. Sample Outcome Audit & Trace Inspection

Displays detailed audit traces for sample claims, highlighting the agent's natural-language reasoning, selected tools, execution mode, and policy explanations.


In [8]:
for result, audit in zip(results[:3], audits[:3]):
    item_demo = {
        "claim_id": result["claim_id"],
        "decision": result["decision"],
        "approved_amount": result["approved_amount"],
        "deducted_amount": result["deducted_amount"],
        "execution_mode": audit["execution_mode"],
        "agent_reasoning": audit.get("agent_reasoning", ""),
        "tools_selected": audit.get("tools_selected_by_agent", []),
        "explanation": result["explanation"]
    }
    print(json.dumps(item_demo, indent=2))


{
  "claim_id": "CLM-001",
  "decision": "APPROVE",
  "approved_amount": 1110.0,
  "deducted_amount": 0.0,
  "execution_mode": "llm_agentic",
  "agent_reasoning": "To audit the claim thoroughly, we need to lookup reimbursement policies for each expense category, verify the receipts for compliance, check the submission timing and thresholds (since it was submitted 8 days after the trip and the total is $1110), and calculate the eligible reimbursement amounts.",
  "tools_selected": [
    "lookup_policy",
    "check_receipts",
    "check_claim_timing_and_thresholds",
    "calculate_reimbursement"
  ],
  "explanation": "All claimed items meet the supplied policy checks. Candidate reimbursement is $1110.00. This is the manager approval tier (POL-APR-02); the recommendation is APPROVE."
}
{
  "claim_id": "CLM-002",
  "decision": "REJECT",
  "approved_amount": 0.0,
  "deducted_amount": 380.0,
  "execution_mode": "llm_agentic",
  "agent_reasoning": "The claim involves personal-sounding expense

## 8. Executive Results Dashboard

A high-contrast visual dashboard summarizing financial aggregates, decision breakdown counts, and quarantined pending manual review amounts.


In [9]:
decision_counts = Counter(result["decision"] for result in results)
approved_total = sum(result["approved_amount"] or 0.0 for result in results)
deducted_total = sum(result["deducted_amount"] or 0.0 for result in results)
pending_claim_ids = [result["claim_id"] for result in results if result["decision"] == "MANUAL_REVIEW"]
pending_total = sum(claim_total(CLAIMS[claim_id]) for claim_id in pending_claim_ids)
rows = "".join(f"<tr style='background:#f7fafc;color:#1a202c'><td style='padding:8px 12px;border:1px solid #cbd5e0'>{decision}</td><td style='padding:8px 12px;border:1px solid #cbd5e0'>{count}</td></tr>" for decision, count in sorted(decision_counts.items()))
mode_summary = ", ".join("{}={}".format(audit["claim_id"], audit["execution_mode"]) for audit in audits)
from IPython.display import HTML, display

display(HTML(f"""
<div style="font-family:Arial,sans-serif;max-width:760px;border:2px solid #4a5568;border-radius:10px;padding:18px;background:#ffffff;color:#1a202c">
<h3 style="margin-top:0;color:#1a202c">Reimbursement results</h3>
<div style="display:flex;gap:12px;flex-wrap:wrap">
<div style="padding:12px;background:#276749;color:#ffffff;border-radius:8px;min-width:180px"><b>Recommended approved</b><br><span style="font-size:1.3em">${approved_total:,.2f}</span></div>
<div style="padding:12px;background:#c05621;color:#ffffff;border-radius:8px;min-width:180px"><b>Recommended deductions</b><br><span style="font-size:1.3em">${deducted_total:,.2f}</span></div>
<div style="padding:12px;background:#2b6cb0;color:#ffffff;border-radius:8px;min-width:180px"><b>Pending Manual Review</b><br><span style="font-size:1.3em">${float(pending_total):,.2f}</span></div>
</div>
<p style="color:#1a202c"><b>Pending claims:</b> {", ".join(pending_claim_ids) if pending_claim_ids else "None"}</p>
<table style="border-collapse:collapse;width:100%"><thead><tr style="background:#2d3748;color:#ffffff"><th style="padding:8px 12px;text-align:left;border:1px solid #4a5568">Decision</th><th style="padding:8px 12px;text-align:left;border:1px solid #4a5568">Claims</th></tr></thead><tbody>{rows}</tbody></table>
<p style="font-size:0.9em;color:#4a5568"><b>Execution modes:</b> {mode_summary}</p>
</div>
"""))







Decision,Claims
APPROVE,1
MANUAL_REVIEW,2
PARTIAL_APPROVE,1
REJECT,1


## 9. Final Structured Deliverable (JSON Schema Array)

The final deliverable cell emits the exact validated JSON array for all 5 claims, strictly adhering to the 9-field assignment schema specification.


In [10]:
print(json.dumps(results, ensure_ascii=False, indent=2, allow_nan=False))






[
  {
    "claim_id": "CLM-001",
    "decision": "APPROVE",
    "approved_amount": 1110.0,
    "deducted_amount": 0.0,
    "missing_docs": [],
    "policy_refs": [
      "POL-AIR-01",
      "POL-APR-02",
      "POL-CAT-01",
      "POL-PD-01",
      "POL-PD-02",
      "POL-RCT-01",
      "POL-TIME-01"
    ],
    "confidence": 0.98,
    "explanation": "All claimed items meet the supplied policy checks. Candidate reimbursement is $1110.00. This is the manager approval tier (POL-APR-02); the recommendation is APPROVE.",
    "tools_used": [
      "lookup_policy",
      "check_receipts",
      "calculate_reimbursement",
      "check_claim_timing_and_thresholds"
    ]
  },
  {
    "claim_id": "CLM-002",
    "decision": "REJECT",
    "approved_amount": 0.0,
    "deducted_amount": 380.0,
    "missing_docs": [],
    "policy_refs": [
      "POL-APR-01",
      "POL-CAT-02",
      "POL-RCT-01",
      "POL-TIME-01"
    ],
    "confidence": 0.98,
    "explanation": "No claimed amount is reimbursable.

## 10. Design Notes, Trade-offs & Engineering Justifications

### 1. Why Separate Cognitive Tool Selection from Mathematical Execution?
Large Language Models are probabilistic sequence predictors. When asked to perform arithmetic on currency, LLMs are vulnerable to subtle hallucinations, rounding inconsistencies, and lack of deterministic reproducibility. In this system:
- The **LLM acts as the cognitive orchestrator**, interpreting unstructured human intent and determining what rules apply.
- **Deterministic Python tools (`Decimal`) execute all financial math**, providing guaranteed repeatability and auditability.

### 2. Why Are `MANUAL_REVIEW` Amounts Nullified?
Under standard accounting and ERP principles, a claim pending manual escalation has not yet received a formal financial determination. Setting `approved_amount = 0.0` could be misinterpreted as a complete rejection, while displaying a candidate amount could lead downstream payment processors to disburse unapproved funds. Setting both amounts to `null` prevents premature disbursement while preserving provisional figures in the internal audit state.

### 3. Fail-Safe Dual-Mode Fallback
Production systems cannot afford downtime caused by 3rd-party LLM outages, rate limits (HTTP 429), or expired API credentials. The LangGraph workflow incorporates an automatic fallback to `deterministic_policy` mode, ensuring that business-critical reimbursement audits continue without interruption.

### 4. Assumptions & Known Limitations
- **Partial Itemization**: The provided dataset supplies trip date spans rather than individual timestamps for each meal or taxi ride. Ambiguous dates within the 30-day window are evaluated safely.
